# CIFAR-10 VGG Standard Training

This notebook contains the training code directly, so Colab runs it without launching a separate Python script.

In [1]:
# Optional: mount Google Drive if your repo lives there.
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
    print(f'Cloned repository to {repo_dir}')
else:
    print(f'Repository already present at {repo_dir}')

Cloned repository to /content/DVBW


In [3]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / 'CIFAR', Path('/content/DVBW/CIFAR'), Path('/content/drive/MyDrive/DVBW/CIFAR')]
WORKDIR = next((path for path in candidates if (path / 'model.py').exists()), None)
if WORKDIR is None:
    raise FileNotFoundError('Could not find the CIFAR folder. Open this notebook from the repo, or set WORKDIR manually.')
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

Working directory: /content/DVBW/CIFAR


In [4]:
%pip install -q matplotlib pillow numpy

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(workers=2, epochs=150, start_epoch=0, train_batch=128, test_batch=128, lr=0.1, drop=0, schedule=[70, 120], gamma=0.1, momentum=0.9, weight_decay=5e-4, checkpoint='./checkpoint/benign/vgg_colab', resume='', manualSeed=1, evaluate=False, gpu_id='0')
args

namespace(workers=2,
          epochs=150,
          start_epoch=0,
          train_batch=128,
          test_batch=128,
          lr=0.1,
          drop=0,
          schedule=[70, 120],
          gamma=0.1,
          momentum=0.9,
          weight_decay=0.0005,
          checkpoint='./checkpoint/benign/vgg_colab',
          resume='',
          manualSeed=1,
          evaluate=False,
          gpu_id='0')

In [6]:
import os
import random
import shutil

import torch
import torch.backends.cudnn as cudnn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from tqdm import tqdm

from model import *
from utils import Logger, AverageMeter, accuracy, mkdir_p, savefig

state = vars(args).copy()
best_acc = 0

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_id
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not available. In Colab, go to Runtime -> Change runtime type -> T4 GPU and rerun.')

random.seed(args.manualSeed)
torch.manual_seed(args.manualSeed)
torch.cuda.manual_seed_all(args.manualSeed)

data_dir = WORKDIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
datasets.CIFAR10(root=str(data_dir), train=True, download=True)
datasets.CIFAR10(root=str(data_dir), train=False, download=True)
print(f'CIFAR-10 is ready in {data_dir}')


def train(model, trainloader, criterion, optimizer):
    model.train()
    losses = AverageMeter(); top1 = AverageMeter(); top5 = AverageMeter()
    pbar = tqdm(total=len(trainloader), desc='Processing')
    for image, target in trainloader:
        image, target = image.cuda(), target.cuda()
        outputs = model(image)
        loss = criterion(outputs, target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        prec1, prec5 = accuracy(outputs.data, target.data, topk=(1, 5))
        losses.update(loss.item(), image.size(0)); top1.update(prec1.item(), image.size(0)); top5.update(prec5.item(), image.size(0))
        pbar.set_postfix({'Epoch': 'train', 'Loss': f'{losses.avg:.4f}', 'top1': f'{top1.avg:.4f}', 'top5': f'{top5.avg:.4f}'})
        pbar.update()
    pbar.close()
    return losses.avg, top1.avg


def test(testloader, model, criterion, split_name='Valid'):
    model.eval()
    losses = AverageMeter(); top1 = AverageMeter(); top5 = AverageMeter()
    pbar = tqdm(total=len(testloader), desc='Processing')
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.cuda(), targets.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            prec1, prec5 = accuracy(outputs.data, targets.data, topk=(1, 5))
            losses.update(loss.item(), inputs.size(0)); top1.update(prec1.item(), inputs.size(0)); top5.update(prec5.item(), inputs.size(0))
            pbar.set_postfix({'Epoch': split_name, 'Loss': f'{losses.avg:.4f}', 'top1': f'{top1.avg:.4f}', 'top5': f'{top5.avg:.4f}'})
            pbar.update()
    pbar.close()
    return losses.avg, top1.avg


def save_checkpoint(state_dict, is_best, checkpoint, filename='checkpoint.pth.tar'):
    filepath = os.path.join(checkpoint, filename)
    torch.save(state_dict, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(checkpoint, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    global state
    if epoch in args.schedule:
        state['lr'] *= args.gamma
        for param_group in optimizer.param_groups:
            param_group['lr'] = state['lr']


def main():
    global best_acc
    start_epoch = args.start_epoch
    if not os.path.isdir(args.checkpoint):
        mkdir_p(args.checkpoint)
    transform_train = transforms.Compose([transforms.RandomHorizontalFlip(), transforms.ToTensor()])
    transform_test = transforms.Compose([transforms.ToTensor()])
    trainset = datasets.CIFAR10(root=str(data_dir), train=True, download=True, transform=transform_train)
    testset = datasets.CIFAR10(root=str(data_dir), train=False, download=True, transform=transform_test)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=args.train_batch, shuffle=True, num_workers=args.workers)
    testloader = torch.utils.data.DataLoader(testset, batch_size=args.test_batch, shuffle=False, num_workers=args.workers)
    print('==> Loading the model')
    model = vgg19_bn()
    model = torch.nn.DataParallel(model).cuda()
    cudnn.benchmark = True
    print('Total params: %.2fM' % (sum(p.numel() for p in model.parameters()) / 1000000.0))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay)
    title = 'CIFAR-10'
    if args.resume:
        checkpoint = torch.load(args.resume)
        best_acc = checkpoint['best_acc']; start_epoch = checkpoint['epoch']
        model.load_state_dict(checkpoint['state_dict']); optimizer.load_state_dict(checkpoint['optimizer'])
        logger = Logger(os.path.join(args.checkpoint, 'log.txt'), title=title, resume=True)
    else:
        logger = Logger(os.path.join(args.checkpoint, 'log.txt'), title=title)
        logger.set_names(['Learning Rate', 'Train Loss', 'Valid Loss', 'Train Acc.', 'Valid Acc.'])
    for epoch in range(start_epoch, args.epochs):
        adjust_learning_rate(optimizer, epoch)
        print('\nEpoch: [%d | %d] LR: %f' % (epoch + 1, args.epochs, state['lr']))
        train_loss, train_acc = train(model, trainloader, criterion, optimizer)
        test_loss, test_acc = test(testloader, model, criterion)
        logger.append([state['lr'], train_loss, test_loss, train_acc, test_acc])
        is_best = test_acc > best_acc
        best_acc = max(test_acc, best_acc)
        save_checkpoint({'epoch': epoch + 1, 'state_dict': model.state_dict(), 'acc': test_acc, 'best_acc': best_acc, 'optimizer': optimizer.state_dict()}, is_best, checkpoint=args.checkpoint)
    logger.close(); logger.plot(); savefig(os.path.join(args.checkpoint, 'log.eps'))
    print('Best acc:'); print(best_acc)


main()

100%|██████████| 170M/170M [00:03<00:00, 42.7MB/s] 


CIFAR-10 is ready in /content/DVBW/CIFAR/data
==> Loading the model
Total params: 20.04M

Epoch: [1 | 150] LR: 0.100000


Processing:   0%|          | 0/391 [00:00<?, ?it/s]

KeyboardInterrupt: 